In [1]:
import polars as pl
import os
from datetime import date, datetime
#from itables import init_notebook_mode
#from itables import show
#init_notebook_mode(all_interactive=True)
#from itables import options
#options.columnDefs = [{"width": "120px", "targets": "_all"}]

In [2]:
ruta_ejecucion_acumulada_vigencias = os.listdir("PIIP/EjecucionProductoAcumuladoVigencias/")
ruta_principal = "PIIP/EjecucionProductoAcumuladoVigencias/"
ejecucion_vigencias = []

for ejecucion in ruta_ejecucion_acumulada_vigencias:
    fuente = pl.read_excel(ruta_principal+ejecucion,table_name="EjecucionPIIP",infer_schema_length=0)
    ejecucion_vigencias.append(fuente)

columnas_ejecucion = ["Bpin","CodigoIndicador","Vigencia","AvanceEjecutado"]
columnas_numericas = ["AvanceEjecutado"]

consolidado_ejecucion = (
    pl.concat(ejecucion_vigencias,how="diagonal").select(columnas_ejecucion)
    .with_columns(
        pl.col(c).cast(pl.Float64)
        for c in columnas_numericas
    ).unique()
)

bpines_reporte = consolidado_ejecucion.select("Bpin").unique()

### POAI - PIIP 2024 - 2026

In [5]:
ruta_poai = "Seguimiento POAI 2026_Corte15Mayo.xlsx"
ruta_proyectos_pi = "ProyectosPI_20260730.xlsx"
columnas_pi = ["Codigo Meta","Código del indicador principal","BPIN","VIGENCIA PI"]
# horizontes = pl.read_excel("PIIP/Reporte_General_Proyectos_2026_20260503.xlsx",has_header=False)

# horizontes = horizontes.rename(dict(list(zip(horizontes.columns,horizontes.row(2))))).slice(3).select("BPIN","Horizonte")
# Omito horizontes, hoy 2026/07/13 PIIP no deja generar el reporte general de proyectos donde se encuentra el estado del proyecto

poai_2026 = pl.read_excel(ruta_poai,table_name="POAI_reporte").drop("Tiene VF")
proyectos_pi = (
    pl.read_excel(ruta_proyectos_pi,table_name="qryProyectos")
    .select(columnas_pi)
    .with_columns(
        pl.col(c).cast(pl.String).str.strip_chars()
        for c in columnas_pi
    ).unique()
)
ruta_proyectos_2024 = "PIIP/ProyectosPIIP_2024/"
ruta_proyectos_2025 = "PIIP/ProyectosPIIP_2025/"
ruta_proyectos_2026 = "PIIP/ProyectosPIIP_2026/"
ruta_regalias = "PIIP/Regalias/"

carpetas = [ruta_proyectos_2024,ruta_proyectos_2025,ruta_proyectos_2026,ruta_regalias]
proyectos_vigencias = []

for carpeta in carpetas:
    datos = os.listdir(carpeta)
    for archivo in datos:
        proyectos_vigencias.append(carpeta+archivo)


indicadores_mga_fichas_piip = []
columnas_indicadores = ["BPIN","ObjetivoEspecifico","CodigoProducto","Producto",
                        "CodigoIndicador","Indicador","Acumula","MetaTotalIndicador","Vigencia","MetaVigencia"]
columnas_info_general = ["BPIN","NombreProyecto","TipoBanco","IdSector","SectorProyecto","Programa","Inicio","Fin",
                         "PoblacionAfectada","PoblacionObjetivo"]
info_general_proyectos = []

duplicado = []

for i in proyectos_vigencias:
    try:
        proyecto_indicadores = pl.read_excel(i,infer_schema_length=0,sheet_name="IProducto")
        info_general = pl.read_excel(i,infer_schema_length=0,sheet_name="InfoGeneral")
        info_general = info_general.transpose()
        info_general = info_general.rename(dict(zip(info_general.columns,info_general.row(0)))).slice(1,)
        indicadores_mga_fichas_piip.append(proyecto_indicadores.select(columnas_indicadores))
        info_general_proyectos.append(info_general.select(columnas_info_general))
        if proyecto_indicadores.filter(pl.col("BPIN").is_in(["202500000032612"])).height > 0:
            duplicado.append(i)
    except ValueError as se: # El valor que me diste es válido como tipo, pero no es aceptable para esta operación
                            # pl.read_excel() recibió un valor de tipo texto valido en su parametro sheet_name, sin embargo,
                            # no se acepta en la operación porque no se encuentra en dataframe
        print(f"El archivo {i} no cuenta con la hoja IProducto")

# Pestaña de Indicadores de los proyectos

indicadores_mga_proyectos = (
    pl.concat(indicadores_mga_fichas_piip).unique()
    .with_columns(pl.col("MetaTotalIndicador").str.replace_many({",":".",".":""}).cast(pl.Float64)) # Aho-corasick para reemplazo simultaneo de una cadena
    .with_columns(pl.col("MetaVigencia").str.replace_many({",":".",".":""}).cast(pl.Float64)) # Aho-corasick para reemplazo simultaneo de una cadena
)# Indicadores programados por vigencia provenientes de las fichas generadas por PIIP

informacion_general_proyectos_piip = pl.concat(info_general_proyectos,how="diagonal").unique()

#indicadores_mga_pivot = []

#for key,value in indicadores_mga_proyectos.group_by("BPIN"):
#    df = (
#        value.select("BPIN","CodigoIndicador").unique()
#        .with_row_index(offset=1,name="Numero")
#        .with_columns(pl.concat_str(pl.lit("Indicador de Producto"),pl.col("Numero")).alias("Numero"))
#        .pivot(index="BPIN",on="Numero")
#    )
#    indicadores_mga_pivot.append(df)

#indicadores_mga_pivot = pl.concat(indicadores_mga_pivot,how="diagonal")

# bpines_unicos = indicadores_mga_proyectos.select("BPIN").unique()

#reporte_pivot = (
#    bpines_unicos
#    .join(indicadores_mga_pivot,on="BPIN",how="inner")
#    .join(informacion_general_proyectos_piip,on="BPIN",how="inner")
#).unique()
# reporte_pivot.write_excel("Reportes Consolidados/ReporteIndicadoresPivot.xlsx",table_name="IndicadoresPivot")

# informacion_general_proyectos_piip.write_excel("Reportes Consolidados/InformacionGeneralProyectos.xlsx",table_name="InformacionGeneral")

"""
3 para sobreejecutados
2 para ejecutados
1 para en ejecucion o parcialmente ejecutados
0 para no ejecutados o no reportados
"""

maxi = pl.col("IndicadorEjecutado").max().over("BPIN")
mini = pl.col("IndicadorEjecutado").min().over("BPIN")

reporte = (
    indicadores_mga_proyectos
    .join(
        consolidado_ejecucion,
        left_on=["BPIN", "CodigoIndicador", "Vigencia"],
        right_on=["Bpin", "CodigoIndicador", "Vigencia"],
        how="left",
        validate="m:1",
    )
    .with_columns(pl.col("AvanceEjecutado").fill_null(0))
    .with_columns(
        pl.when(pl.col("AvanceEjecutado") == 0).then(pl.lit(0))
        .when(pl.col("AvanceEjecutado") > pl.col("MetaVigencia")).then(pl.lit(3))
        .when(pl.col("AvanceEjecutado") == pl.col("MetaVigencia")).then(pl.lit(2))
        .otherwise(pl.lit(1))
        .alias("IndicadorEjecutado")
    )
    .with_columns(
        pl.when(mini == 3).then(pl.lit("Proyecto SobreEjecutado"))
        .when((maxi == 3) & (mini == 2)).then(pl.lit("Proyecto con uno o mas Indicadores SobreEjecutados"))
        .when(mini == 2).then(pl.lit("Proyecto Ejecutado"))
        .when(maxi >= 1).then(pl.lit("Proyecto Ejecutado Parcialmente"))
        .otherwise(pl.lit("Proyecto No Ejecutado - No Reportado"))
        .alias("TipoEjecucion")
    ).drop("IndicadorEjecutado")
)

# reporte.write_excel("Reportes Consolidados/ReporteProgramacionEjecucion_20260511.xlsx",table_name="ProgramacionEjecucionProyectos")

# EjecucionFisicaProyectos = (
#     reporte
#     .write_excel(
#         "Proyectos Ejecutados/TipoEjecucionProyectosPIIP_20260516.xlsx",
#         table_name="ProyectosEjecutados",
#         table_style="Table Style Light 11",
#         autofit=True,
#     )
# )

El archivo PIIP/Regalias/Reporte Ficha actualizada del proyecto_07072026162650.xlsx no cuenta con la hoja IProducto


In [3]:
ruta_poai = "Seguimiento POAI 2026_Corte30julio.xlsx"
ruta_proyectos_pi = "ProyectosPI_20260730.xlsx"
columnas_pi = ["Codigo Meta","Código del indicador principal","BPIN","VIGENCIA PI"]

poai_2026 = pl.read_excel(ruta_poai,table_name="POAI_reporte").drop("Tiene VF")
proyectos_pi = (
    pl.read_excel(ruta_proyectos_pi,table_name="qryProyectos")
    .select(columnas_pi)
    .with_columns(
        pl.col(c).cast(pl.String).str.strip_chars()
        for c in columnas_pi
    ).unique()
)
ruta_proyectos_vigencia_2026 = "PIIP/ProyectosPIIP_GESPROY_2026_20260729/"

proyectos_vigencia_2026 = []

for archivo in os.listdir(ruta_proyectos_vigencia_2026):
    proyectos_vigencia_2026.append(ruta_proyectos_vigencia_2026+archivo)




columnas_indicadores = ["BPIN","ObjetivoEspecifico","CodigoProducto","Producto",
                        "CodigoIndicador","Indicador","Acumula","MetaTotalIndicador","Vigencia","MetaVigencia"]
columnas_info_general = ["BPIN","NombreProyecto","TipoBanco","IdSector","SectorProyecto","Programa","Inicio","Fin",
                         "PoblacionAfectada","PoblacionObjetivo"]

indicadores_mga_ficha_proyecto = []
informacion_general_proyectos_2026 = []

for i in proyectos_vigencia_2026:
    try:
        info_general = pl.read_excel(i,infer_schema_length=0,sheet_name="InfoGeneral")
        info_general = info_general.transpose()
        info_general = info_general.rename(dict(zip(info_general.columns,info_general.row(0)))).slice(1,)
        informacion_general_proyectos_2026.append(info_general.select(columnas_info_general))
    except ValueError as ig:
        print(f"El archivo del BPIN {info_general.select("BPIN")} no cuenta con la hoja InfoGeneral")
    try:
        proyecto_indicadores = pl.read_excel(i,infer_schema_length=0,sheet_name="IProducto")
        indicadores_mga_ficha_proyecto.append(proyecto_indicadores.select(columnas_indicadores))
    except ValueError as ip:
        print(f"El archivo del BPIN {info_general.select("BPIN").row(0)[0]} no cuenta con la hoja IProducto")

indicadores_proyectos_2026 = (
    pl.concat(indicadores_mga_ficha_proyecto).unique()
    .with_columns(pl.col("MetaTotalIndicador").str.replace_many({",":".",".":""}).cast(pl.Float64))
    .with_columns(pl.col("MetaVigencia").str.replace_many({",":".",".":""}).cast(pl.Float64))
)

informacion_general_proyectos_2026 = pl.concat(informacion_general_proyectos_2026,how="diagonal").unique()

"""
3 para sobreejecutados
2 para ejecutados
1 para en ejecucion o parcialmente ejecutados
0 para no ejecutados o no reportados
"""

maxi = pl.col("IndicadorEjecutado").max().over("BPIN")
mini = pl.col("IndicadorEjecutado").min().over("BPIN")

reporte_proyectos_2026 = (
    informacion_general_proyectos_2026
    .join(indicadores_proyectos_2026,on=["BPIN"],how="inner")
    .join(
        consolidado_ejecucion,
        left_on=["BPIN", "CodigoIndicador", "Vigencia"],
        right_on=["Bpin", "CodigoIndicador", "Vigencia"],
        how="left",
        validate="m:1",
    )
    .with_columns(pl.col("AvanceEjecutado").fill_null(0))
    .filter(pl.when( pl.max_horizontal("MetaVigencia","AvanceEjecutado") == 0 ).then(pl.lit(False)).otherwise(pl.lit(True)))
    .with_columns(
        pl.when(pl.col("AvanceEjecutado") == 0).then(pl.lit(0))
        .when(pl.col("AvanceEjecutado") > pl.col("MetaVigencia")).then(pl.lit(3))
        .when(pl.col("AvanceEjecutado") == pl.col("MetaVigencia")).then(pl.lit(2))
        .otherwise(pl.lit(1))
        .alias("IndicadorEjecutado")
    )
    .with_columns(
        pl.when(mini == 3).then(pl.lit("Proyecto SobreEjecutado"))
        .when((maxi == 3) & (mini == 2)).then(pl.lit("Proyecto con uno o mas Indicadores SobreEjecutados"))
        .when(mini == 2).then(pl.lit("Proyecto Ejecutado"))
        .when(maxi >= 1).then(pl.lit("Proyecto Ejecutado Parcialmente"))
        .otherwise(pl.lit("Proyecto No Ejecutado - No Reportado"))
        .alias("TipoEjecucion")
    ).drop("IndicadorEjecutado")
)

Could not determine dtype for column 5, falling back to string


El archivo del BPIN 2023002700175 no cuenta con la hoja IProducto
El archivo del BPIN 2022002700077 no cuenta con la hoja IProducto


### Poblaciones

In [ ]:
poblaciones = (
    informacion_general_proyectos_piip
    .select("BPIN","PoblacionAfectada","PoblacionObjetivo")
    .unique()
    .with_columns(pl.col("PoblacionAfectada").cast(pl.Int64),pl.col("PoblacionObjetivo").cast(pl.Int64))
)

In [ ]:
actividades_proyectos_fichas_piip = []
columnas_actividades = ["BPIN","NombreProyecto","ObjetivoEspecifico","CodigoIndicador","Indicador","Acumula","MetaTotalIndicador","Vigencia","MetaVigencia","EntregableActividad",
                        "NombreEntregableActividad","ValorSolicitado","ValorVigente"]

for i in proyectos_vigencias:
    proyecto_actividades = pl.read_excel(i,infer_schema_length=0,sheet_name="CActividad")
    proyecto_actividades = proyecto_actividades.select(columnas_actividades)
    actividades_proyectos_fichas_piip.append(proyecto_actividades)

municipios_proyectos_fichas_piip = []

for i in proyectos_vigencias:
    municipios_proyectos = pl.read_excel(i,infer_schema_length=0,sheet_name="Localizacion")
    municipios_proyectos = municipios_proyectos.select("BPIN","Departamento","Municipio")
    municipios_proyectos_fichas_piip.append(municipios_proyectos)

fuentes_financiacion_proyectos_fichas_piip = []
columnas_fuentes_financiacion = ["BPIN","NombreProyecto","TipoEntidad","Entidad","TipoRecurso",
            "Vigencia","Solicitado","Inicial","Vigente"]

for i in proyectos_vigencias:
    try:
        fuentes_proyectos = pl.read_excel(
            i,
            infer_schema_length=0,
            sheet_name="FFinanciacion"
        ).select(columnas_fuentes_financiacion)
        fuentes_financiacion_proyectos_fichas_piip.append(fuentes_proyectos)

    except pl.exceptions.NoDataError:
        print(f"[{i}] Hoja vacía")
    except pl.exceptions.ColumnNotFoundError as e:
        print(f"[{i}] Falta columna: {e}")
    except ValueError as e:
        # Si no tiene recursos asignados esta hoja no va a existir en la ficha
        schema_vacio = {c: pl.Utf8 for c in columnas_fuentes_financiacion}
        fuentes_proyectos = pl.DataFrame(schema=schema_vacio)

fuentes_proyectos_fichas_piip = []
columnas_regionalizacion = ["BPIN","NombreProyecto","TipoFinanciador","EntidadFinanciadora","TipoRecurso","CodigoIndicador","Indicador",
                           "MetaTotalIndicador","Vigencia","MetaVigencia","Departamento","Municipio","TipoAgrupacion","Agrupacion",
                           "ValorRegionalizado","ValorRegionalizadoSolicitado","ValorRegionalizadoVigente","MetaRegionalizada",]


for j in proyectos_vigencias:
    try:
        regionalizacion_proyecto = pl.read_excel(i,infer_schema_length=0,sheet_name="Regionalizacion")
        regionalizacion_proyecto = regionalizacion_proyecto.select(columnas_regionalizacion)
        fuentes_proyectos_fichas_piip.append(regionalizacion_proyecto)
    except pl.exceptions.NoDataError:
        print(f"[{j}] hoja vacía")
    except pl.exceptions.ColumnNotFoundError as e:
        print(f"[{j}] falta columna: {e}")
    except ValueError as e:
        schema_vacio = {c: pl.Utf8 for c in columnas_regionalizacion}
        fuentes_proyectos = pl.DataFrame(schema=schema_vacio)

# Pestaña de actividades de los proyectos

actividades_proyectos = (
    pl.concat(actividades_proyectos_fichas_piip)
    .with_columns(pl.col("MetaTotalIndicador").cast(pl.Float64))
    .with_columns(pl.col("MetaVigencia").cast(pl.Float64))
    .with_columns(pl.col("ValorSolicitado").cast(pl.Float64))
    .with_columns(pl.col("ValorVigente").cast(pl.Float64))
    .join(horizontes,on="BPIN",how="left")# Hago un cast sencillo, a diferencia de indicadores proyecto, 
                                                        # esta tabla no muestra comas para representar miles
) # Indicadores programados por vigencia provenientes de las fichas generadas por PIIP

# Pestaña de municipios de los proyectos

municipios_proyectos = (
    pl.concat(municipios_proyectos_fichas_piip)
    .with_columns(pl.col("Municipio").filter(pl.col("Municipio")!="").str.join(", ").over("BPIN").alias("Municipios"))
    .with_columns(
        pl.when(pl.col("Departamento")!="Sucre").then(pl.lit("Otro"))
        .when(
                (pl.col("Departamento")=="Sucre") &
                (pl.col("Municipios").is_null()) | (pl.col("Municipios")=="")
        ).then(pl.lit("Todo el Departamento de Sucre")).otherwise(pl.col("Municipios")).alias("Municipios")
    )
    #.join(horizontes,on="BPIN",how="left") # Omitido por el momento
    .select("BPIN","Departamento","Municipios","Horizonte").unique()
) # Municipios por proyecto provenientes de las fichas generadas por PIIP

# Pestaña de regionalización de los proyectos

regionalizacion_proyectos = (
    pl.concat(fuentes_proyectos_fichas_piip)
    .with_columns(
        pl.when(pl.col("Departamento")!="Sucre").then(pl.lit("Otros"))
        .when(
                pl.col("Departamento")=="Sucre",
                ~pl.col("Municipio").is_null(),
                pl.col("Municipio")==""
        ).then(pl.lit("Todo el Departamento de Sucre")).otherwise("Municipio").alias("Municipio")
    )
    .with_columns(pl.col("MetaTotalIndicador").cast(pl.Float64))
    .with_columns(pl.col("MetaVigencia").cast(pl.Float64))
    .with_columns(pl.col("ValorRegionalizadoSolicitado").cast(pl.Float64))
    .with_columns(pl.col("ValorRegionalizadoVigente").cast(pl.Float64))
    .with_columns(pl.col("MetaRegionalizada").cast(pl.Float64))
    #.join(horizontes,on="BPIN",how="left") # Hago un cast sencillo, a diferencia de indicadores proyecto, 
    # Omitido por el momento                # esta tabla no muestra comas para representar miles
)

# Pestaña fuentes de financiación de los proyectos

fuente_financiacion_proyectos = (
    pl.concat(fuentes_financiacion_proyectos_fichas_piip)
    .with_columns(pl.col("Solicitado").cast(pl.Float64))
    .with_columns(pl.col("Inicial").cast(pl.Float64))
    .with_columns(pl.col("Vigente").cast(pl.Float64))
    #.join(horizontes,on="BPIN",how="left")
)

# poai_2026 = poai_2026.join(indicadores_mga_proyectos,on="BPIN").with_columns(pl.lit(0).alias("Ejecución 2025"),pl.lit("POAI 2026").alias("Categoría"))

# regionalizacion_proyectos.write_excel("Plataformas/RegionalizacionProyectos_20260507.xlsx",table_name="Regionalizacion")

"""
Tipos de joins que se pueden realizar:
    indicadores_mga_proyectos y actividades_proyectos con las llaves BPIN, Codigo Indicador y vigencia.
    indicadores_mga_proyectos y actividades_proyectos con municipio usando la llave BPIN.
    fuente_financiacion_proyectos y regionalizacion_proyectos usando BPIN, tipo de recurso, vigencia. Regionalización podría servir como detalle de fuentes
    indicadores_mga_proyectos y actividades_proyectos con regionalizacion_proyectos usando BPIN, Codigo Indicador y vigencia.
""" 

# municipios_proyectos.write_excel("Proyectos Ejecutados/MunicipiosProyectos_20260517.xlsx",table_name="Municipios",autofit=True)

### Procesos en PIIP

In [ ]:
ruta_procesos_piip = "PIIP/reporte_procesos_piip_20260507.xlsx"

columnas_procesos_piip = [
    "ID", "BPIN", "Nombre del proyecto", "Estado del Proyecto",
    "Proceso", "Estado Proceso",
    "Fecha de Inicio del Proceso", "Fecha Fin del Proceso",
    "Paso actual", "Fecha de Inicio del Paso", "Fecha Fin del Paso",
    "Nombre proceso trámite","Estado proceso trámite","Fecha fin trámite"
]
columnas_fecha = [
    "Fecha de Inicio del Proceso", "Fecha Fin del Proceso",
    "Fecha de Inicio del Paso", "Fecha Fin del Paso","Fecha fin trámite"
]

procesos_piip = (
    pl.read_excel(ruta_procesos_piip, table_name="ProcesosPIIP")
    .select(columnas_procesos_piip)
    .with_columns([
        pl.col(c).str.to_datetime(
            "%d-%m-%Y %H:%M:%S",
            time_zone="America/Bogota",
            strict=False,
        )
        for c in columnas_fecha
    ])
    # Renombrar la columna original para posterior comparativa con fechas. Puede que no coincida estado de PIIP con estado calculado
    .rename({"Estado del Proyecto": "Estado del Proyecto (PIIP)"})
)

def fecha_ultimo_tramite(proceso: str, nombre_tramite: str) -> pl.Expr:
    return (
        pl.when(
            (pl.col("Proceso") == proceso)
            & (pl.col("Estado Proceso")=="Completado")
            & (pl.col("Nombre proceso trámite") == nombre_tramite)
            & (pl.col("Fecha Fin del Paso").is_not_null())
            & (pl.col("Estado proceso trámite") == "Completado")
        ).then(pl.col("Fecha fin trámite"))
        .otherwise(None)
        .max()
        .over("ID")
    )

def numero_ajustes(proceso: str, paso_contiene: str) -> pl.Expr:
    return (
        pl.when(
            (pl.col("Proceso") == proceso)
            & (pl.col("Estado Proceso")=="Completado")
            & (pl.col("Paso actual").str.contains(paso_contiene))
            & (pl.col("Fecha Fin del Paso").is_not_null())
            & (pl.col("Estado proceso trámite") == "Completado")
        )
        .then(pl.lit(1))
        .otherwise(None)
        .sum()
        .over("ID")
    )


# fecha de cierre del paso final de un subproceso, por proyecto. Me puedo apoyar despues con coalesce
def fecha_cierre_paso(proceso: str, paso_contiene: str) -> pl.Expr:
    # Verifica el proceso tengo el ultimo paso y que tenga registro en la fecha de fin de paso. Luego agrupa con over por ID y devuelve la fecha mas reciente
    return (
        pl.when(
            (pl.col("Proceso") == proceso)
            & (pl.col("Estado Proceso")=="Completado")
            & (pl.col("Paso actual").str.contains(paso_contiene))
            & (pl.col("Fecha Fin del Paso").is_not_null())
        )
        .then(pl.col("Fecha Fin del Paso"))
        .otherwise(None)
        .max()
        .over("ID")
    )


# Calcular fechas de transición de estados
proyectos_anotados = procesos_piip.with_columns([
    # Formulado -> Viable: viabilidad nativa de PIIP o migración desde SUIFP
    # El primer valor no nulo retornado de la función fecha_cierre_paso. Hay dos posibilidades, migró de suifp o es proyecto nuevo de PIIP 
    pl.coalesce([
        fecha_cierre_paso("Viabilidad ET", "Firma y emisión"),
        fecha_cierre_paso("Migración viabilidad TERRITORIO", "Ficha"),
    ]).alias("Fecha Formulado -> Viable"),

    # Viable -> Viable con recursos solicitados
    fecha_cierre_paso("Solicitud de recursos ET", "Firma y emisión")
        .alias("Fecha Viable -> Viable con recursos solicitados"),

    # Viable con recursos solicitados -> En Ejecución
    # Es la firma del Director del proyecto en LB1 (o LB0 si LB1 aún no existe).
    # Tomamos el max para quedarnos con el último que se firmó.
    # Debo tener presente que el proceso de reporte de avance de ejecución puede estar activo sin estos dos procesos
    pl.coalesce(
        pl.max_horizontal([
            fecha_cierre_paso("Planear La Ejecución - LB1", "Aprobación director del proyecto"),
            fecha_cierre_paso("Planear La Ejecución - LB0", "Aprobación director del proyecto"),
        ]),
        fecha_cierre_paso("Reporte avance ejecución", "Reporte avance")
    ).alias("Fecha Viable con recursos solicitados -> En Ejecución"),

    numero_ajustes("Ajustes con trámite presupuestal ET","Pendiente de aprobación").alias("Número ajustes con trámite presupuestal ET completados"),
    numero_ajustes("Ajustes sin trámite presupuestal ET","Firmas y emisión").alias("Número ajustes sin trámite presupuestal ET completados"),
    fecha_ultimo_tramite("Ajustes con trámite presupuestal ET","Decreto de liquidación ET").alias("Decreto de liquidación ET"),
    fecha_ultimo_tramite("Ajustes con trámite presupuestal ET","SALDOS TERRITORIO").alias("SALDOS TERRITORIO"),
    fecha_ultimo_tramite("Ajustes con trámite presupuestal ET","Trámite adición ET").alias("Trámite adición ET"),
    fecha_ultimo_tramite("Ajustes con trámite presupuestal ET","Trámite reducción ET").alias("Trámite reducción ET"),
    
    # ¿El proyecto es migrado de SUIFP?
    (pl.col("Proceso") == "Migración viabilidad TERRITORIO")
        .any().over("ID").alias("Es Migrado SUIFP"),
])

# 4. Colapsar a una fila por proyecto
cambios_estado_proyectos = (
    proyectos_anotados
    .group_by("ID")
    .agg([
        pl.col("BPIN").cast(pl.Int64).max(),
        pl.col("Nombre del proyecto").first(),
        pl.col("Estado del Proyecto (PIIP)").first(),
        pl.col("Es Migrado SUIFP").first(),
        pl.col("Fecha Formulado -> Viable").first(),
        pl.col("Fecha Viable -> Viable con recursos solicitados").first(),
        pl.col("Fecha Viable con recursos solicitados -> En Ejecución").first(),
        pl.col("Número ajustes con trámite presupuestal ET completados").first(),
        pl.col("Número ajustes sin trámite presupuestal ET completados").first(),
        pl.col("Decreto de liquidación ET").first(),
        pl.col("SALDOS TERRITORIO").first(),
        pl.col("Trámite adición ET").first(),
        pl.col("Trámite reducción ET").first(),
    ])
)

# Estado del Proyecto calculado, incluyendo el caso "Formulado"
cambios_estado_proyectos = cambios_estado_proyectos.with_columns([
    pl.when(pl.col("Fecha Viable con recursos solicitados -> En Ejecución").is_not_null())
        .then(pl.lit("En Ejecución"))
    .when(pl.col("Fecha Viable -> Viable con recursos solicitados").is_not_null())
        .then(pl.lit("Viable con recursos solicitados"))
    .when(pl.col("Fecha Formulado -> Viable").is_not_null())
        .then(pl.lit("Viable"))
    .otherwise(pl.lit("Formulado"))
    .alias("Estado del Proyecto (calculado)"),
])

# Tiempos entre transiciones de estados
hoy = pl.lit(date.today())
cambios_estado_proyectos = cambios_estado_proyectos.with_columns([
    # Días entre transiciones
    (pl.col("Fecha Viable -> Viable con recursos solicitados").cast(pl.Date)
     - pl.col("Fecha Formulado -> Viable").cast(pl.Date))
        .dt.total_days()
        .alias("Días Viable -> Viable con recursos solicitados"),

    (pl.col("Fecha Viable con recursos solicitados -> En Ejecución").cast(pl.Date)
     - pl.col("Fecha Viable -> Viable con recursos solicitados").cast(pl.Date))
        .dt.total_days()
        .alias("Días Viable con recursos solicitados -> En Ejecución"),

    # Días totales de todo el ciclo de formulación -> En Ejecución (solo para los que llegaron)
    (pl.col("Fecha Viable con recursos solicitados -> En Ejecución").cast(pl.Date)
     - pl.col("Fecha Formulado -> Viable").cast(pl.Date))
        .dt.total_days()
        .alias("Días totales del ciclo (Formulado -> En Ejecución)"),

    # Días desde la última transición hasta hoy: cuánto lleva en el estado actual
    (hoy - pl.coalesce([
        pl.col("Fecha Viable con recursos solicitados -> En Ejecución"),
        pl.col("Fecha Viable -> Viable con recursos solicitados"),
        pl.col("Fecha Formulado -> Viable"),
    ]).cast(pl.Date)).dt.total_days()
        .alias("Días en estado actual"),

    # ¿Coincide el estado de PIIP con el calculado?
    (pl.col("Estado del Proyecto (calculado)") != pl.col("Estado del Proyecto (PIIP)"))
        .alias("Discrepancia con PIIP"),
])

cambios_estado_proyectos = cambios_estado_proyectos.with_columns([
    pl.col("BPIN").cast(pl.String),
    pl.col("Fecha Formulado -> Viable").cast(pl.Date),
    pl.col("Fecha Viable -> Viable con recursos solicitados").cast(pl.Date),
    pl.col("Fecha Viable con recursos solicitados -> En Ejecución").cast(pl.Date),
    pl.col("Decreto de liquidación ET").cast(pl.Date),
    pl.col("SALDOS TERRITORIO").cast(pl.Date),
    pl.col("Trámite adición ET").cast(pl.Date),
    pl.col("Trámite reducción ET").cast(pl.Date)
])

# cambios_estado_proyectos.write_excel(
#     "ReporteProcesosPrueba_20260507_v4.xlsx",
#     table_name="ProcesosProyectos",
# )

### Reporte Avance Actividades

In [ ]:
ruta_avance_actividades = "PIIP/Reporte_Avance_Actividades_Marzo2026_20260503.xlsx"

columnas_actividades = ["BPIN","Nombre_Proyecto","Objetivo_Especifico","Actividad","Fecha_Inicio","Fecha_Fin","UM","Complemento_UM",
                        "Apropiacion_Inicial__Actividad","Apropiacion_Vigente__Actividad","Costo_Total_Actividad","Cantidad_Total_Actividad","Costo_Unitario_Actividad",
                        "Cantidad_Programada_Para_El_Mes","Cantidad_Ejecutada_Para_El_Mes","Observacion_Cantidades","Costo_Ejecutado_Para_El_Mes",
                        "Costo_Ejecutado_Para_El_Mes_Recursos_Vigentes","Costo_Ejecutado_Para_El_Mes_Reservas_Presupuestales","Observacion_Presupuestal",
                        "Costo_Ejecutado_Para_El_Mes_Flujo_De_Caja","Costo_Ejecutado_Para_El_Mes_Flujo_Caja_Recursos_Vigentes",
                        "Costo_Ejecutado_Para_El_Mes_Flujo_Caja_Reservas_Presupuestales","Observacion_Flujo_De_Caja"]

columnas_para_cast = ["Apropiacion_Inicial__Actividad","Apropiacion_Vigente__Actividad","Costo_Total_Actividad","Costo_Unitario_Actividad",
                      "Cantidad_Programada_Para_El_Mes","Cantidad_Ejecutada_Para_El_Mes","Costo_Ejecutado_Para_El_Mes","Costo_Ejecutado_Para_El_Mes_Recursos_Vigentes",
                      "Costo_Ejecutado_Para_El_Mes_Reservas_Presupuestales","Costo_Ejecutado_Para_El_Mes_Flujo_De_Caja","Costo_Ejecutado_Para_El_Mes_Flujo_Caja_Recursos_Vigentes",
                     "Costo_Ejecutado_Para_El_Mes_Flujo_Caja_Reservas_Presupuestales"]

avance_actividades = (
    pl.read_excel(ruta_avance_actividades,infer_schema_length=0).select(columnas_actividades)
    .with_columns([
        pl.col(c).str.replace_many({",":".",".":""}).cast(pl.Float64)
        for c in columnas_para_cast
    ])
)

### Reporte Regionalizado

In [ ]:
ruta_reginalizado_acumulado = "PIIP/Reporte_Proyecto seguimiento productos regionalizado acumulado_2026_20260503.xlsx"

columnas_regionalizacion = ["Bpin","NombreProyecto","ObjetivoEspecifico","CodigoIndicador","Indicador","UMI","MetaTotalIndicador","AvanceTotal",
                           "Vigencia","MetaVigenciaIndicador","PeriodoAcumulado","AvanceVigencia","TipoEntidad","EntidadFinanciadora","Recurso",
                            "Departamento","Municipio","TipoAgrupacion","Agrupacion","Solicitado","Vigente","Compromiso","Obligacion","Pago","MetaVigenteReg",
                           "AvanceMetaReg","DepDivipola","MunDivipola"]

# pl.read_excel(ruta_reginalizado_acumulado).select(columnas_regionalizacion)

### Fuentes de Financiación

In [4]:
from xlsxwriter import Workbook

seleccion_columnas = ["PROYECTO","NOMBRE_RECURSO","APROPIACION_DEFINITIVA","RP","PAGO","VIGENCIA"]

fuentes_hacienda = {
    "2022": "SIIFWEB/EjecucionGastosHaciendaCUIPO/EjecucionGastos2022_HACIENDA_Enero-Diciembre.xlsx",
    "2023": "SIIFWEB/EjecucionGastosHaciendaCUIPO/EjecucionGastos2023_HACIENDA_Enero-Diciembre.xlsx",
    "2024": "SIIFWEB/EjecucionGastosHaciendaCUIPO/EjecucionGastos2024_HACIENDA_Enero-Diciembre.xlsx",
    "2025": "SIIFWEB/EjecucionGastosHaciendaCUIPO/EjecucionGastos2025_HACIENDA_Enero-Diciembre.xlsx",
    "2026": "SIIFWEB/EjecucionGastosHaciendaCUIPO/EjecucionGastos2026_HACIENDA_1Enero-14Julio.xlsx",
}

ff_vigencias = []

for vigencia, ruta in fuentes_hacienda.items():
    ff = (
        pl.read_excel(ruta, table_name=f"hacienda_{vigencia}", infer_schema_length=0)
        .with_columns(
            pl.col("PROYECTO").str.strip_chars(),
            pl.col("APROPIACION_DEFINITIVA").cast(pl.Float64),
            pl.col("RP").cast(pl.Float64),
            pl.col("PAGO").cast(pl.Float64),
            pl.lit(vigencia).alias("VIGENCIA")
        )
        .filter(pl.col("PROYECTO") != "", pl.col("PROYECTO") != "0")
        .select(seleccion_columnas)
    )
    ff_vigencias.append(ff)

fuentes_consolidadas = pl.concat(ff_vigencias, how="diagonal")

nombre_columnas = {"PROYECTO":"bpin","NOMBRE_RECURSO":"fuente","APROPIACION_DEFINITIVA":"apropiacion definifiva",
                   "RP":"rp","PAGO":"pago","VIGENCIA":"vigencia",
                   "AL MENOS UN COMPROMISO EN LA VIGENCIA":"al menos un compromiso","AL MENOS UN PAGO EN LA VIGENCIA":"al menos un pago"}
    

calculo_fuentes = (
    fuentes_consolidadas
    .with_columns(
        # (pl.col("RP")-pl.col("PAGO")).alias("RP - PAGO"),
        pl.when(pl.col("RP").mean().over("PROYECTO") > 0)
        .then(pl.lit(True)).otherwise(pl.lit(False)).alias("AL MENOS UN COMPROMISO EN LA VIGENCIA"),
        pl.when(pl.col("PAGO").mean().over(["PROYECTO"]) > 0)
        .then(pl.lit(True)).otherwise(False).alias("AL MENOS UN PAGO EN LA VIGENCIA"))
    .with_columns(
        pl.col("PROYECTO").str.head(8).alias("ES BPIN").str.to_integer(strict=False),
        pl.col("PROYECTO").str.len_chars().alias("LARGO BPIN"))
    .filter(
        ~pl.col("ES BPIN").is_null(),
        pl.col("LARGO BPIN")>7)
    .drop("ES BPIN","LARGO BPIN")
    .unique()
    .rename(nombre_columnas)
    .group_by("bpin","fuente","vigencia").agg(
        pl.col("apropiacion definifiva").sum(),pl.col("rp").sum(),
        pl.col("pago").sum(),pl.col("al menos un compromiso").unique().str.join(","),pl.col("al menos un pago").unique().str.join(",")
    )
)

exportar = (
    calculo_fuentes
    .write_excel(
        "Proyectos Ejecutados/FuenteFinanciacionProyectos_20260715.xlsx",
        table_name="EjecucionFinanciera",
        autofit=True,
        table_style={
            "style" : "Table Style Light 11",
            "first_column" : True
        },
        column_formats={
            "id": {"font": "Aptos Narrow"},
            "points": {"align": "center"},
            "z-score": {"align": "center"},
        },
    )
)

## SIIFWEB

In [4]:
"""RUTAS"""

dfs = {
    "cdps" : {
        "2022":"SIIFWEB/Informes/Consolidado disponibilidades/SinFormatearEnExcel (Tal como se descarga desde SIIFWEB)/consolidado_cert_dispw_2022_1Enero_31Diciembre.xlsx",
        "2023":"SIIFWEB/Informes/Consolidado disponibilidades/SinFormatearEnExcel (Tal como se descarga desde SIIFWEB)/consolidado_cert_dispw_2023_1Enero_31Diciembre.xlsx",
        "2024":"SIIFWEB/Informes/Consolidado disponibilidades/SinFormatearEnExcel (Tal como se descarga desde SIIFWEB)/consolidado_cert_dispw_2024_1Enero_31Diciembre.xlsx",
        "2025":"SIIFWEB/Informes/Consolidado disponibilidades/SinFormatearEnExcel (Tal como se descarga desde SIIFWEB)/consolidado_cert_dispw_2025_1Enero_31Diciembre.xlsx",
        "2026":"SIIFWEB/Informes/Consolidado disponibilidades/20260729/consolidado_cert_dispw_20260729.xlsx"
    },
    "compromisos" : {
        "2022":"SIIFWEB/Informes/Consolidado compromisos/SinFormatearEnExcel (Tal como se descarga desde SIIFWEB)/consolidado_compromisosw_2022_202607181700.xlsx",
        "2023":"SIIFWEB/Informes/Consolidado compromisos/SinFormatearEnExcel (Tal como se descarga desde SIIFWEB)/consolidado_compromisosw_2023_202607181700.xlsx",
        "2024":"SIIFWEB/Informes/Consolidado compromisos/SinFormatearEnExcel (Tal como se descarga desde SIIFWEB)/consolidado_compromisosw_2024_202607181700.xlsx",
        "2025":"SIIFWEB/Informes/Consolidado compromisos/SinFormatearEnExcel (Tal como se descarga desde SIIFWEB)/consolidado_compromisosw_2025_202607181700.xlsx",
        "2026":"SIIFWEB/Informes/Consolidado compromisos/20260729/consolidado_compromisosw_20260729.xlsx",
    },
    "obligaciones" : {
        "2022":"SIIFWEB/Informes/Consolidado obligaciones/SinFormatearEnExcel (Tal como se descarga desde SIIFWEB)/consolidado_obligacionw_2022_20260730.xlsx",
        "2023":"SIIFWEB/Informes/Consolidado obligaciones/SinFormatearEnExcel (Tal como se descarga desde SIIFWEB)/consolidado_obligacionw_2023_20260730.xlsx",
        "2024":"SIIFWEB/Informes/Consolidado obligaciones/SinFormatearEnExcel (Tal como se descarga desde SIIFWEB)/consolidado_obligacionw_2024_20260730.xlsx",
        "2025":"SIIFWEB/Informes/Consolidado obligaciones/SinFormatearEnExcel (Tal como se descarga desde SIIFWEB)/consolidado_obligacionw_2025_20260730.xlsx",
        "2026":"SIIFWEB/Informes/Consolidado obligaciones/20260729/consolidado_obligacionw_20260729.xlsx",
    },
    "reservas" : {
        "2024":"SIIFWEB/Informes/Consolidado reservas/SinFormatearEnExcel (Tal como se descarga desde SIIFWEB)/consolidado_reservaw_2024_20260730.xlsx",
        "2025":"SIIFWEB/Informes/Consolidado reservas/SinFormatearEnExcel (Tal como se descarga desde SIIFWEB)/consolidado_reservaw_2025_20260730.xlsx",
        "2026":"SIIFWEB/Informes/Consolidado reservas/SinFormatearEnExcel (Tal como se descarga desde SIIFWEB)/consolidado_reservaw_2026_corte30Julio_20260730.xlsx",
    },
    "comprobantes" : {
        "2022":"SIIFWEB/Informes/Comprobante por centro de costo/SinFormatearEnExcel (Tal como se descarga desde SIIFWEB)/aux_comprobantes32w_2022_20260730.xlsx",
        "2023":"SIIFWEB/Informes/Comprobante por centro de costo/SinFormatearEnExcel (Tal como se descarga desde SIIFWEB)/aux_comprobantes32w_2023_20260730.xlsx",
        "2024":"SIIFWEB/Informes/Comprobante por centro de costo/SinFormatearEnExcel (Tal como se descarga desde SIIFWEB)/aux_comprobantes32w_2024_20260730.xlsx",
        "2025":"SIIFWEB/Informes/Comprobante por centro de costo/SinFormatearEnExcel (Tal como se descarga desde SIIFWEB)/aux_comprobantes32w_2025_20260730.xlsx",
        "2026":"SIIFWEB/Informes/Comprobante por centro de costo/SinFormatearEnExcel (Tal como se descarga desde SIIFWEB)/aux_comprobantes32w_2026_corte30Julio_20260730.xlsx",
    },
    "historico_gasto":{
        "2017_2026":"SIIFWEB/Informes/Historial de orden de gasto 2/20260729/historial_ordengasto2_20260729.xlsx"
    }
}

### Funciones

In [5]:
def procesar_float(df, *columna_valor) -> pl.Expr:
    return (
        df.with_columns(
            pl.col(col).str.replace_many({",":"."}).cast(pl.Float64).alias(col)
            for col in columna_valor
        )
    )

def procesar_date(df, *columna_fecha) -> pl.Expr:
    return(
        df.with_columns(
            pl.when( pl.col(col).str.len_chars() == 8 )
            .then( pl.col(col).str.strptime(pl.Date,"%d/%m/%y",strict=False) )
            .when( pl.col(col).str.len_chars() == 10 )
            .then( pl.col(col).str.strptime(pl.Date,"%d/%m/%Y",strict=False) )
            .otherwise(pl.lit(None,dtype=pl.Date))
            .alias(col)
            for col in columna_fecha
        )
    )

def procesar_int(df, columnas_numericas_int) -> pl.Expr:
    lectura_vigencia = df.with_columns(
        pl.col(columnas_numericas_int).cast(pl.Int32),
    )
    return lectura_vigencia

#### CDPs

In [7]:
lista_consolidado_cdps = []

columnas_numericas_cdp = [
    "VALOR_CERTIFICADO",
    "VALOR_DISPONIBILIDAD_DEF",
    "VALOR_COMPROMISOS",
    "SALDO_CERTF",
    "OBLIGACIONES",
    "PAGOS"
]

columnas_fecha_cdp = ["FECHA_DISP",]

columnas_cdp = [
    "NRO_CDP",
    "VIGENCIA",
    "VALOR_CERTIFICADO",
    "VALOR_DISPONIBILIDAD_DEF",
    "VALOR_COMPROMISOS",
    "SALDO_CERTF",
    "OBLIGACIONES",
    "PAGOS"
]

for key, value in dfs.items():
    if key == "cdps":
        for vigencia, ruta in value.items():
            lectura_vigencia = pl.read_excel(ruta,infer_schema_length=0)
            lectura_vigencia = lectura_vigencia.with_columns(pl.lit(int(vigencia)).alias("VIGENCIA"))
            lectura_vigencia = procesar_float(lectura_vigencia,*columnas_numericas_cdp)
            lectura_vigencia = procesar_date(lectura_vigencia,*columnas_fecha_cdp)
            lista_consolidado_cdps.append(lectura_vigencia)

"""
1. Un mismo CDP puede provenir de diferentes rubros
2. Un CDP puede tener varios proyectos
3. Varios proyectos pueden tomar de un mismo CDP y un proyecto puede tomar de varios CDP
4. Voy a encontrar CDPs duplicados pero porque diferentes proyectos toman de ese CDP en diferentes fechas
5. .with_columns(pl.col("FECHA_DISP").min().over("NRO_CDP").alias("PRIMERA_FECHA_DISP")) no puedo hacer esto porque el numero del cdp se
   genera desde 1 en cada vigencia. Agrupar un CDP agruparia el mismo para diferentes vigencias
"""

cdps_por_proyecto = (
    # Cdps generados proyecto
    pl.concat(lista_consolidado_cdps)
    .filter(pl.col("PROYECTO")!="0")
    .sort(by=["NRO_CDP","VIGENCIA"],descending=False)
)

consolidado_cdps_por_vigencia = (
    # Ver los valores totales de los CDP generados en una vigencia
    pl.concat(lista_consolidado_cdps)
    .select(columnas_cdp)
    .group_by("NRO_CDP","VIGENCIA").agg(pl.col(columnas_numericas_cdp).sum())
)

cdps_por_proyecto = (
    cdps_por_proyecto
    .sort(by=[pl.col("NRO_CDP").cast(pl.Int32),"VIGENCIA"],descending=False)
    .group_by("PROYECTO","VIGENCIA").agg(
        pl.col("FECHA_DISP").min().alias("FechaPrimerCDP"),
        pl.col("VALOR_CERTIFICADO").sum().alias("ValorCertificado"),
        pl.col("VALOR_DISPONIBILIDAD_DEF").sum().alias("ValorDisponibilidadDefinitiva")
    )
)

In [9]:
"""
A continuación presento una serie de calculos que confirman mi siguiente analisis de la base de datos de siifweb
1. Un CDP puede amparar varios proyectos de inversión.
2. Un proyecto puede ejecutarse con cargo a varios CDPs: la financiación de un proyecto puede estar respaldada por múltiples 
   certificados de disponibilidad, expedidos a lo largo de la vigencia
3. Un CDP puede afectar varios rubros presupuestales
4. La apropiación de un rubro puede ser afectada sucesivamente por múltiples CDPs, hasta agotar su saldo de apropiación disponible

El CDP es el instrumento mediante el cual se afecta preliminarmente la apropiación presupuestal. En su expedición, 
un mismo certificado puede amparar uno o varios proyectos de inversión y afectar uno o varios rubros y fuentes de financiación; 
a su vez, un proyecto puede ejecutarse con cargo a múltiples CDPs y la apropiación de un rubro puede ser afectada por sucesivos certificados. En consecuencia, la relación entre proyectos, certificados de disponibilidad y rubros presupuestales es de muchos a muchos, y el nivel mínimo de desagregación de la información es la imputación presupuestal: la combinación CDP–rubro–fuente

La apropiación se establece a nivel de rubro presupuestal. El CDP es el instrumento mediante el cual se afecta preliminarmente 
esa apropiación, garantizando la existencia de recursos para asumir un compromiso. Un mismo CDP puede amparar varios proyectos 
y afectar varios rubros y fuentes de financiación; a su vez, un proyecto puede ejecutarse con cargo a múltiples CDPs, sea 
porque concurre más de una fuente de financiación —cada una con su propia disponibilidad— o porque el proyecto se ejecuta 
mediante compromisos sucesivos a lo largo de la vigencia.

"""

comprobacion_cdp_proyectos = (
    # Con esto confirmo que un CDP puede contener varios proyectos.
    pl.concat(lista_consolidado_cdps).group_by("NRO_CDP","VIGENCIA")
    .agg(pl.col("PROYECTO").unique()).with_columns(pl.col("PROYECTO").list.len().alias("NUMERO PROYECTOS"))
    .sort(by="NUMERO PROYECTOS",descending=True)
)

comprobacion_proyectos_cdp = (
    # Con esto confirmo que un proyecto puede tomar de varios CDPs
    pl.concat(lista_consolidado_cdps).filter(pl.col("PROYECTO")!="0").group_by("PROYECTO","VIGENCIA")
    .agg(pl.col("NRO_CDP").unique()).with_columns(pl.col("NRO_CDP").list.len().alias("NUMERO CDP"))
    .sort(by="NUMERO CDP",descending=True)
)

comprobacion_cdp_rubro = (
    # Con esto confirmo que un cdp puede contener varios rubros
    pl.concat(lista_consolidado_cdps).group_by("NRO_CDP","VIGENCIA")
    .agg(pl.col("NOMBRE_RUBRO").unique()).with_columns(pl.col("NOMBRE_RUBRO").list.len().alias("NUMERO RUBROS"))
    .sort(by="NUMERO RUBROS",descending=True)
)

comprobacion_rubro_cdp = (
    # Con esto confirmo que un rubro puede contener varios cdps
    pl.concat(lista_consolidado_cdps).group_by("NOMBRE_RUBRO","VIGENCIA")
    .agg(pl.col("NRO_CDP").unique()).with_columns(pl.col("NRO_CDP").list.len().alias("NUMERO CDP"))
    .sort(by="NUMERO CDP",descending=True)
)

comprobacion_rubro_fuente = (
    # Con esto confirmo que un rubro puede contener varias fuentes
    pl.concat(lista_consolidado_cdps).group_by("NOMBRE_RUBRO","VIGENCIA")
    .agg(pl.col("FONDO").unique()).with_columns(pl.col("FONDO").list.len().alias("FONDOS"))
    .sort(by="FONDOS",descending=True)
)

comprobacion_fuente_rubro = (
    # Con esto confirmo que una fuente puede pertenecer a diferentes rubros
    pl.concat(lista_consolidado_cdps).group_by("NOMBRE_FONDO","VIGENCIA")
    .agg(pl.col("NOMBRE_RUBRO").unique()).with_columns(pl.col("NOMBRE_RUBRO").list.len().alias("RUBROS"))
    .sort(by="RUBROS",descending=True)
)

comprobacion_cdp_fecha = (
    # Con esto confirmo que un cdp contiene una fecha unico de expedición
    pl.concat(lista_consolidado_cdps).group_by("NRO_CDP","VIGENCIA")
    .agg(pl.col("FECHA_DISP").unique()).with_columns(pl.col("FECHA_DISP").list.len().alias("FECHAS"))
    .sort(by="FECHAS",descending=True).filter(pl.col("FECHAS")>1)
)

#### Compromisos

In [8]:
lista_consolidado_compromisos = []

columnas_fechas_compromisos = ['FECHA_REG',]

columnas_numericas_compromisos = ['VALOR_REGISTRO','VALOR_COMPROMISO_DEF','SALDO_RP','OBLIGACIONES','PAGOS']

for key, value in dfs.items():
    if key == "compromisos":
        for vigencia, ruta in value.items():
            lectura_vigencia = pl.read_excel(ruta,infer_schema_length=0)
            lectura_vigencia = lectura_vigencia.with_columns(pl.lit(int(vigencia)).alias("VIGENCIA"))
            lectura_vigencia = procesar_float(lectura_vigencia,*columnas_numericas_compromisos)
            lectura_vigencia = procesar_date(lectura_vigencia,*columnas_fechas_compromisos)
            lista_consolidado_compromisos.append(lectura_vigencia)

consolidado_compromisos = (
    pl.concat(lista_consolidado_compromisos)
    .filter(pl.col("PROYECTO")!="0")
)

conteo_rp_proyecto = (
    consolidado_compromisos
    .group_by("NRO_RP").agg(pl.col("PROYECTO").unique())
    .with_columns(pl.col("PROYECTO").list.len())
    .sort(by=["PROYECTO"],descending=True)
    .row(0)
)

print(f"""
De acuerdo con la tabla de compromisos un rp puede contener varios proyectos. 
Veamos: el rp {conteo_rp_proyecto[0]} cuenta con {conteo_rp_proyecto[1]} proyectos""")


rps_por_proyecto = (
    consolidado_compromisos
    .group_by("PROYECTO","VIGENCIA").agg(
        pl.col("FECHA_REG").min().alias("FechaPrimerRP"),
        pl.col("VALOR_REGISTRO").sum().alias("ValorRegistro"),
        pl.col("VALOR_COMPROMISO_DEF").sum().alias("ValorCompromisoDefinitivo")
    )
)


De acuerdo con la tabla de compromisos un rp puede contener varios proyectos. 
Veamos: el rp 5289 cuenta con 11 proyectos


#### Obligaciones

In [9]:
lista_consolidado_obligaciones = []

columnas_fechas_obligaciones = ["FECHA_OBLI",]

columnas_numericas_obligaciones = ["VALOR_OBLIGACION","VALOR_OBLI_DEF","SALDO_OBLI","PAGOS"]

for key, value in dfs.items():
    if key == "obligaciones":
        for vigencia, ruta in value.items():
            lectura_vigencia = pl.read_excel(ruta,infer_schema_length=0)
            lectura_vigencia = lectura_vigencia.with_columns(pl.lit(int(vigencia)).alias("VIGENCIA"))
            lectura_vigencia = procesar_float(lectura_vigencia,*columnas_numericas_obligaciones)
            lectura_vigencia = procesar_date(lectura_vigencia,*columnas_fechas_obligaciones)
            lista_consolidado_obligaciones.append(lectura_vigencia)

consolidado_obligaciones = (
    pl.concat(lista_consolidado_obligaciones)
    .filter(pl.col("PROYECTO")!="0")
)


obligacion_por_proyecto = (
    consolidado_obligaciones
    .group_by("PROYECTO","VIGENCIA").agg(
        pl.col("TIPO_ORDEN_GASTO").unique().str.join(", "),
        pl.col("FECHA_OBLI").min().alias("FechaPrimeraObligacion"),
        pl.col("VALOR_OBLIGACION").sum().alias("ValorObligacion"),
        pl.col("VALOR_OBLI_DEF").sum().alias("ValorObligacionDefinitiva"),
        pl.col("SALDO_OBLI").sum().alias("ValorSaldoObligacion"),
        pl.col("PAGOS").sum().alias("ValorPagos")
    )
)

#### Reservas

In [10]:
lista_consolidado_reservas = []

columnas_fechas_reservas = ["FECHA_RESERVA","FECHA_CDP"]

columnas_numericas_reservas = ["VALOR_RESERVA","VALOR_RESERVA_DEF","SALDO_RESERVA","OBLIGACIONES_RESERVA","PAGOS_RESERVA"]

for key, value in dfs.items():
    if key == "reservas":
        for vigencia, ruta in value.items():
            lectura_vigencia = pl.read_excel(ruta,infer_schema_length=0)
            lectura_vigencia = lectura_vigencia.with_columns(pl.lit(int(vigencia)).alias("VIGENCIA"))
            lectura_vigencia = procesar_float(lectura_vigencia,*columnas_numericas_reservas)
            lectura_vigencia = procesar_date(lectura_vigencia,*columnas_fechas_reservas)
            lista_consolidado_reservas.append(lectura_vigencia)

consolidado_reservas = (
    pl.concat(lista_consolidado_reservas)
    .filter(pl.col("PROYECTO")!="0")
)

reservas_por_proyecto = (
    consolidado_reservas
    .group_by("PROYECTO","VIGENCIA").agg(
        pl.col("FECHA_RESERVA").min().alias("FechaReserva"),
        pl.col("VALOR_RESERVA").sum().alias("ValorReserva"),
        pl.col("VALOR_RESERVA_DEF").sum().alias("ValorReservaDefinitiva"),
        pl.col("OBLIGACIONES_RESERVA").sum().alias("ObligacionesReservasDefinitivas"),
        pl.col("SALDO_RESERVA").sum().alias("ValorSaldoReservas"),
        pl.col("PAGOS_RESERVA").sum().alias("ValorPagoReservas")
    )
)

#### Comprobantes

In [11]:
lista_consolidado_comprobantes = []

columnas_fechas_comprobantes = ["FECHA"]

columnas_numericas_comprobantes = ["VALOR"]

for key, value in dfs.items():
    if key == "comprobantes":
        for vigencia, ruta in value.items():
            lectura_vigencia = pl.read_excel(ruta,infer_schema_length=0)
            lectura_vigencia = lectura_vigencia.with_columns(pl.lit(int(vigencia)).alias("VIGENCIA"))
            lectura_vigencia = procesar_float(lectura_vigencia,*columnas_numericas_comprobantes)
            lectura_vigencia = procesar_date(lectura_vigencia,*columnas_fechas_comprobantes)
            lista_consolidado_comprobantes.append(lectura_vigencia)

consolidado_comprobantes = (
    pl.concat(lista_consolidado_comprobantes)
    .filter(pl.col("PROYECTO")!="0",pl.col("TIPO_COMPPTAL")=="CONTRACREDITO")
)

contracredito_por_proyecto = (
    consolidado_comprobantes
    .group_by("PROYECTO","VIGENCIA").agg(
        pl.col("FECHA").min().alias("FechaPrimerContracredito"),
        pl.col("VALOR").sum().alias("ValorContraceditos")
    ).join(cdps_por_proyecto.filter(pl.col("ValorDisponibilidadDefinitiva")==0),on=["PROYECTO","VIGENCIA"],validate="1:1")
    .drop("FechaPrimerCDP","ValorCertificado","ValorDisponibilidadDefinitiva")
) # Tener presente que un proyecto puede tener varios contracreditos

#### Historico de gasto

In [12]:
lista_historico_ordenes_gasto = []

columnas_entero_historico_orden_gasto = ["PREFIJO"]

columnas_numericas_historico_orden_gasto = ["VALOR_CONTRATO","VALOR_PAGO"]

columnas_fechas_historico_orden_gasto = ["FECHA_FIRMA","FECHA_FINAL1","FECHA_INICIO","FECHA_PAGO"]

for key, value in dfs.items():
    if key == "historico_gasto":
        for vigencia, ruta in value.items():
            lectura_vigencia = pl.read_excel(ruta,infer_schema_length=0)
            lectura_vigencia = procesar_float(lectura_vigencia,*columnas_numericas_historico_orden_gasto)
            lectura_vigencia = procesar_date(lectura_vigencia,*columnas_fechas_historico_orden_gasto)
            lectura_vigencia = procesar_int(lectura_vigencia,*columnas_entero_historico_orden_gasto)
            lista_historico_ordenes_gasto.append(lectura_vigencia)

historico_ordenes_gasto_contratos = (
    pl.concat(lista_historico_ordenes_gasto)
    .filter(pl.col("PROYECTO")!="0")
)

contratos_por_proyecto = (
    historico_ordenes_gasto_contratos
    .group_by("PROYECTO","PREFIJO").agg(
        pl.col("FECHA_FIRMA").min().alias("FechaFirmaPrimerContrato"),
        pl.col("TIPO_CONTRATO").len().alias("NumeroContratos"),
        pl.col("VALOR_CONTRATO").sum().alias("ValorContratos")
    ).rename({"PREFIJO":"VIGENCIA"})
)

### Comprobaciones

In [60]:
reportes_sutec = [
    cdps_por_proyecto,
    consolidado_compromisos,
    consolidado_obligaciones,
    consolidado_ordenes_gasto,
    consolidado_reservas,
    consolidado_comprobantes,
    historico_ordenes_gasto_contratos
]

# -------------------------------------------------- CDP - COMPROMISOS -------------------------------------------------------------------

"""
El nivel de granularidad se refiere al grado de detalle o la profundidad con la que se registran, analizan o presentan los datos. 
A mayor detalle, mayor es la granularidad (o "grano fino"), y a menor detalle (datos agrupados o resumidos), menor es la granularidad 
(o "grano grueso")
"""

disponibilidades = (
    cdps_por_proyecto.group_by("VIGENCIA","NRO_CDP").agg(
        pl.col("VALOR_DISPONIBILIDAD_DEF").sum(),
        # La fecha de expedición es única por (VIGENCIA, NRO_CDP) — verificado en 2022-2026, cero excepciones — así que .first() es seguro
        pl.col("FECHA_DISP").first()
    )
)

compromisos_segun_cdp = (
    # Agrupo a nivel de proyecto para tener un nivel de granularidad de compromisos a nivel de proyecto, no de cdo
    # por que se usan todas estas llaves? Vayamos desglozando por llave: 
    # 1. VIGENCIA: Teniendo en cuenta que el CDP se reinicia desde 1 cada vigencia
    # 2. IDENTIFICACION_PRESUPUESTAL: Un CDP puede afectar varios rubros presupuestales
    # 3. FONDO: Un CDP puede afectar varias fuentes de financiación
    # 4. NRO_CDP: identifica el documento dentro de la vigencia
    # 5. PROYECTO: un CDP puede amparar varios proyectos; sin esta llave, sus porciones se suman como si fueran una sola
    # estas cinco columnas definen el grano imputación × proyecto; con menos llaves se mezclan valores de grupos distintos
    cdps_por_proyecto.group_by("VIGENCIA","IDENTIFICACION_PRESUPUESTAL","FONDO","NRO_CDP","PROYECTO").agg(
        pl.col("VALOR_COMPROMISOS").sum()
    )
)

compromisos_por_proyecto = (
    consolidado_compromisos.group_by("VIGENCIA","IDENTIFICACION_PRESUPUESTAL","FONDO","NRODOC_CDP","PROYECTO").agg(
        pl.col("VALOR_COMPROMISO_DEF").sum()
    )
)

comprobacion_disponibilidad_compromiso = (
    # Conciliación: VALOR_COMPROMISOS en disponibilidades es un acumulado arrastrado desde compromisos. 
    # Si las llaves y el grano están bien, el cruce debe dar diferencia cero en toda fila. Verificado: cuadra al peso en las 5 vigencias
    compromisos_segun_cdp
    .join(
        compromisos_por_proyecto,
        left_on=["VIGENCIA","IDENTIFICACION_PRESUPUESTAL","FONDO","NRO_CDP","PROYECTO"],
        right_on=["VIGENCIA","IDENTIFICACION_PRESUPUESTAL","FONDO","NRODOC_CDP","PROYECTO"],
        how="full",
        coalesce=True
    )
    .with_columns((pl.col("VALOR_COMPROMISOS").fill_null(0)-pl.col("VALOR_COMPROMISO_DEF").fill_null(0)).alias("COMP"))
    .filter(pl.col("COMP").abs()>=1)
)


# -------------------------------------------------- COMPROMISOS - OBLIGACIONES -------------------------------------------------------------------

obligaciones_segun_rp = (
    consolidado_compromisos.group_by("VIGENCIA","NRO_RP","IDENTIFICACION_PRESUPUESTAL","FONDO","PROYECTO").agg(
        pl.col("OBLIGACIONES").sum(),
        pl.col("FECHA_REG").first()
    )
)

obligaciones_por_vigencia = (
    consolidado_obligaciones.group_by("VIGENCIA","NRO_RP","IDENTIFICACION_PRESUPUESTAL","FONDO","PROYECTO").agg(
        pl.col("VALOR_OBLI_DEF").sum()
    )
)

comprobacion_obligacion = (
    # Conciliación: OBLIGACIONES en compromisos es un acumulado arrastrado desde obligaciones. 
    # Si las llaves y el grano están bien, el cruce debe dar diferencia cero en toda fila. Verificado: cuadra al peso en las 5 vigencias,
    # salvo 2023, donde el consolidado de obligaciones incluye 35 obligaciones de reservas (RPs de 2022) por $7.613 millones
    obligaciones_segun_rp.join(
        obligaciones_por_vigencia,
        left_on=["VIGENCIA","NRO_RP","IDENTIFICACION_PRESUPUESTAL","FONDO","PROYECTO"],
        right_on=["VIGENCIA","NRO_RP","IDENTIFICACION_PRESUPUESTAL","FONDO","PROYECTO"],
        how="full",
        coalesce=True
    ).with_columns((pl.col("OBLIGACIONES").fill_null(0)-pl.col("VALOR_OBLI_DEF").fill_null(0)).alias("OBLI"))
    .filter(pl.col("OBLI").abs()>=1)
)

comprobacion_reserva = (
    # Este codigo me da las obligaciones que se causaron con cargo a reservas — es decir, la ejecución de las reservas, no su constitución. 
    # Cada fila huérfana confirmada contra v−1 es "esta cuenta se recibió y causó en la vigencia v, contra un compromiso que quedó 
    # reservado del año v−1". Para 2023, eso reconstruye la ejecución de reservas que el sistema no reportó por ningún otro lado: es tu único 
    # registro de ese flujo.
    obligaciones_por_vigencia.join(
        obligaciones_segun_rp,
        left_on=["VIGENCIA","NRO_RP","IDENTIFICACION_PRESUPUESTAL","FONDO","PROYECTO"],
        right_on=["VIGENCIA","NRO_RP","IDENTIFICACION_PRESUPUESTAL","FONDO","PROYECTO"],
        how="anti",
    ).join(
        obligaciones_segun_rp,
        left_on=[pl.col("VIGENCIA")-1,"NRO_RP","IDENTIFICACION_PRESUPUESTAL","FONDO","PROYECTO"],
        right_on=["VIGENCIA","NRO_RP","IDENTIFICACION_PRESUPUESTAL","FONDO","PROYECTO"],
        how="left",
        suffix="_rp_anterior"
    ).with_columns(
        pl.col("OBLIGACIONES").is_null().alias("anomalia"),
        (pl.col("PROYECTO") != pl.col("PROYECTO_rp_anterior")).alias("proyecto_difiere"),
    ).join(
        compromisos_segun_cdp.group_by("VIGENCIA","IDENTIFICACION_PRESUPUESTAL","FONDO","PROYECTO").agg(pl.col("NRO_CDP").unique().alias("CDPs_d")),
        left_on=[pl.col("VIGENCIA")-1,"IDENTIFICACION_PRESUPUESTAL","FONDO","PROYECTO"],
        right_on=["VIGENCIA","IDENTIFICACION_PRESUPUESTAL","FONDO","PROYECTO"],
        how="left",
        suffix="_cdp"
    ).join(
        consolidado_reservas.group_by("VIGENCIA","IDENTIFICACION_PRESUPUESTAL","FONDO","PROYECTO").agg(pl.col("NUMERO_CDP").unique().alias("CDPs_r")),
        left_on=["VIGENCIA","IDENTIFICACION_PRESUPUESTAL","FONDO","PROYECTO"],
        right_on=["VIGENCIA","IDENTIFICACION_PRESUPUESTAL","FONDO","PROYECTO"],
        how="left"
    ).with_columns(pl.col("CDPs_d").list.set_intersection(pl.col("CDPs_r")).alias("CDPs_r_d"))
    .with_columns(pl.when(pl.col("CDPs_r_d").list.len()>0).then(pl.lit(True)).otherwise(pl.lit(False)).alias("coincidencia_cdps"))
    # Tiene sentido que todo sea False porque no hay reporte de reserva para 2023
)


# -------------------------------------------------- OBLIGACIONES - ORDENES DE GASTO-------------------------------------------------------------------


obligaciones_segun_orden_gasto = (
    consolidado_obligaciones.group_by("VIGENCIA","NRO_ORDEN_GASTO").agg(
        pl.col("VALOR_OBLI_DEF").sum()
    )
)

obligaciones_segun_contrato = (
    consolidado_ordenes_gasto.group_by("VIGENCIA","NRO_CONTRATO").agg(
        pl.col("TOTAL").sum()
    )
)

comprobacion_obliaciones_ordenes = (
    obligaciones_segun_contrato
    .join(
        obligaciones_segun_orden_gasto,
        left_on=["VIGENCIA","NRO_CONTRATO"],
        right_on=["VIGENCIA","NRO_ORDEN_GASTO"],
        how="left",
        coalesce=True
    )
)


# -------------------------------------------------- COMPROMISOS - HISTORICO GASTO -------------------------------------------------------------------

historico_ordenes_gasto_contratos_agrupado = (
    historico_ordenes_gasto_contratos.group_by("PREFIJO","NRODOC","RUBRO","RECURSO","PROYECTO").agg(
        pl.col("VALOR_CONTRATO").sum()
    )
)

comprobacion_contratos_compromisos = (
    obligaciones_segun_rp
    .join(
        historico_ordenes_gasto_contratos_agrupado,
        left_on=["VIGENCIA","NRO_RP","IDENTIFICACION_PRESUPUESTAL","FONDO","PROYECTO"],
        right_on=["PREFIJO","NRODOC","RUBRO","RECURSO","PROYECTO"],
        how="left"
    )
)

NameError: name 'cdps_por_proyecto' is not defined

### Analisis

In [193]:
from datetime import date

disponibilidad_def_segun_cdp = (
    cdps_por_proyecto.group_by("VIGENCIA","IDENTIFICACION_PRESUPUESTAL","FONDO","NRO_CDP","PROYECTO").agg(
        pl.col("VALOR_DISPONIBILIDAD_DEF").sum(),
        pl.col("FECHA_DISP").first()
    )
)

compromisos_por_proyecto = (
    consolidado_compromisos.group_by("VIGENCIA","IDENTIFICACION_PRESUPUESTAL","FONDO","NRODOC_CDP","PROYECTO").agg(
        pl.col("VALOR_COMPROMISO_DEF").sum(),
    )
)

agregado_disponibilidades_compromisos = (
    disponibilidad_def_segun_cdp
    .join(
        compromisos_por_proyecto,
        left_on=["VIGENCIA","IDENTIFICACION_PRESUPUESTAL","FONDO","NRO_CDP","PROYECTO"],
        right_on=["VIGENCIA","IDENTIFICACION_PRESUPUESTAL","FONDO","NRODOC_CDP","PROYECTO"],
        how="inner",
    )
    .with_columns((pl.col("VALOR_DISPONIBILIDAD_DEF")-pl.col("VALOR_COMPROMISO_DEF")).alias("SALDO_SIN_COMPROMETER"))
)

disponibilidad_sin_compromiso = (
    compromisos_por_proyecto
    .join(
        disponibilidad_def_segun_cdp,
        left_on=["VIGENCIA","IDENTIFICACION_PRESUPUESTAL","FONDO","NRODOC_CDP","PROYECTO"],
        right_on=["VIGENCIA","IDENTIFICACION_PRESUPUESTAL","FONDO","NRO_CDP","PROYECTO"],
        how="anti",
    )
)

#if disponibilidad_sin_compromiso.height == 0:
#    print(f"""De acuerdo con las llaves y el nivel de granularidad de la tabla de disponibilidades y compromisos 
#    todos los CDPs soportan al menos un RP. Vease {disponibilidad_sin_compromiso} 
#    el cual realiza un join tipo anti desde compromisos a disponibilidades""")
#else:
#    print(f"""De acuerdo con las llaves y el nivel de granularidad de la tabla de disponibilidades y compromisos 
#    hay algunos RPs que no están soportados por CDPs. Vease: {disponibilidad_sin_compromiso} 
#    el cual realiza un join tipo anti desde compromisos a disponibilidades""")

ColumnNotFoundError: unable to find column "IDENTIFICACION_PRESUPUESTAL"; valid columns: ["PROYECTO", "VIGENCIA", "FechaPrimerCDP", "ValorCertificado", "ValorDisponibilidadDefinitiva"]

Resolved plan until failure:

	---> FAILED HERE RESOLVING 'group_by' <---
DF ["PROYECTO", "VIGENCIA", "FechaPrimerCDP", "ValorCertificado", ...]; PROJECT */5 COLUMNS

This error occurred with the following context stack:
	[1] 'group_by'


## Reporte

In [13]:
consolidado_siifweb = (
    cdps_por_proyecto
    .join(rps_por_proyecto,on=["PROYECTO","VIGENCIA"],how="left",validate="1:1")
    .join(obligacion_por_proyecto,on=["PROYECTO","VIGENCIA"],how="left",validate="1:1")
    .join(reservas_por_proyecto,left_on=["PROYECTO","VIGENCIA"],right_on=["PROYECTO",pl.col("VIGENCIA")-1],how="left",validate="1:1")
    .drop("PROYECTO_right","VIGENCIA_right")
    .join(contracredito_por_proyecto,on=["PROYECTO","VIGENCIA"],how="left",validate="1:1")
    .join(contratos_por_proyecto,on=["PROYECTO","VIGENCIA"],how="left",validate="1:1")
)

consolidado_siifweb.write_excel("SIIFWEB/Reportes/ReporteSIIFWEB.xlsx",table_name="ReporteSIIFWEB")